# 2차 — 멀티시드 견고 탐색 (Robust Hyperparameter Search)

지금까지 Optuna 탐색은 항상 **단일 시드**(보통 3-fold, random_state 고정)로
평가했습니다. 그런데 시드별 OOF 표준편차가 0.00011이나 된다는 걸 직접
측정했었죠 — 즉, 단일 시드 탐색은 "운 좋게 그 시드에서만 잘 나온" 파라미터를
최고로 뽑을 위험이 있습니다.

이 노트북은 **탐색 자체를 3개 시드 × 3-fold(=9번 평가)로 평균내서**, 특정
시드에 운 좋게 맞아떨어진 파라미터가 아니라 **여러 시드에서 일관되게
좋은 파라미터**를 찾습니다. 그다음 그 파라미터로 10시드 배깅까지 돌려서,
기존 `best_params.json`(단일시드 탐색 결과)의 배깅 결과(0.74036)와
직접 비교합니다.

torch 없이 LightGBM만 사용합니다.

**예상 시간**: 탐색(30trial × 3seed × 3fold = 270회) + 최종 검증(10seed × 5fold
= 50회) 합쳐서 30~60분 예상.

**저장 위치**: 이 노트북은 `experiment_history/2차/`, 공유 자원은
`experiment_history/` 바로 아래, 제출 후보 CSV는 `experiment_history/submit file/`에
저장합니다 (실제 제출 전까지는 `submission file/`에 넣지 않습니다).

## 1. 라이브러리, 설정, 기존 best_params 로드 (비교용)

In [1]:
import os
import json
import numpy as np
import pandas as pd
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = "../../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

SEARCH_SEEDS = [1004, 42, 7]   # 탐색 단계에서 평균낼 시드 3개
SEARCH_SPLITS = 3              # 시드당 fold 수 (속도 확보)
N_TRIALS = 30

FINAL_SEEDS = [1004, 42, 7, 123, 2024, 88, 555, 999, 31, 77]  # 최종 비교용 10시드
FINAL_SPLITS = 5

CACHE_DIR = "../blend_cache"
SUBMIT_DIR = "../submit file"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(SUBMIT_DIR, exist_ok=True)

with open("../best_params.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
old_best_params = loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

print("기존(단일시드 탐색) best_params:")
print(json.dumps(old_best_params, indent=2, ensure_ascii=False))

기존(단일시드 탐색) best_params:
{
  "n_estimators": 2357,
  "learning_rate": 0.013668973845707234,
  "num_leaves": 159,
  "max_depth": 5,
  "min_child_samples": 83,
  "subsample": 0.908634361540321,
  "colsample_bytree": 0.7944647062780377,
  "reg_alpha": 6.529095336314542,
  "reg_lambda": 0.00029237017648296726,
  "min_split_gain": 0.4557413392057567
}


## 2. 데이터 전처리 (main.py와 동일)

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

df_le = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col].astype(str))

y = df_le[TARGET_COL].values.astype(np.float32)
X = df_le.drop(columns=[TARGET_COL])

print(f"전처리 완료: {X.shape}")

전처리 완료: (256351, 67)


## 3. 멀티시드 견고 탐색 (3시드 × 3-fold = 9회 평균, 30 trial)

In [3]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
    }

    all_scores = []
    for seed in SEARCH_SEEDS:
        skf = StratifiedKFold(n_splits=SEARCH_SPLITS, shuffle=True, random_state=seed)
        for tr_idx, val_idx in skf.split(X, y):
            m = LGBMClassifier(**params, random_state=seed, verbose=-1)
            m.fit(X.iloc[tr_idx], y[tr_idx])
            preds = m.predict_proba(X.iloc[val_idx])[:, 1]
            all_scores.append(roc_auc_score(y[val_idx], preds))

    return float(np.mean(all_scores))


def print_progress(study, trial):
    print(f"Trial {trial.number + 1}/{N_TRIALS}  평균AUC={trial.value:.5f}  (현재 최고: {study.best_value:.5f})")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS, callbacks=[print_progress])

robust_best_params = study.best_params
print("\n견고 탐색 최적 파라미터:")
print(json.dumps(robust_best_params, indent=2, ensure_ascii=False))
print(f"탐색 단계 평균 AUC (3시드x3fold): {study.best_value:.5f}")

Trial 1/30  평균AUC=0.73898  (현재 최고: 0.73898)
Trial 2/30  평균AUC=0.73894  (현재 최고: 0.73898)
Trial 3/30  평균AUC=0.73821  (현재 최고: 0.73898)
Trial 4/30  평균AUC=0.73964  (현재 최고: 0.73964)
Trial 5/30  평균AUC=0.73870  (현재 최고: 0.73964)
Trial 6/30  평균AUC=0.73806  (현재 최고: 0.73964)
Trial 7/30  평균AUC=0.73685  (현재 최고: 0.73964)
Trial 8/30  평균AUC=0.73967  (현재 최고: 0.73967)
Trial 9/30  평균AUC=0.73896  (현재 최고: 0.73967)
Trial 10/30  평균AUC=0.73887  (현재 최고: 0.73967)
Trial 11/30  평균AUC=0.72816  (현재 최고: 0.73967)
Trial 12/30  평균AUC=0.73964  (현재 최고: 0.73967)
Trial 13/30  평균AUC=0.73968  (현재 최고: 0.73968)
Trial 14/30  평균AUC=0.73960  (현재 최고: 0.73968)
Trial 15/30  평균AUC=0.73966  (현재 최고: 0.73968)
Trial 16/30  평균AUC=0.73956  (현재 최고: 0.73968)
Trial 17/30  평균AUC=0.73938  (현재 최고: 0.73968)
Trial 18/30  평균AUC=0.73932  (현재 최고: 0.73968)
Trial 19/30  평균AUC=0.73962  (현재 최고: 0.73968)
Trial 20/30  평균AUC=0.73944  (현재 최고: 0.73968)
Trial 21/30  평균AUC=0.73920  (현재 최고: 0.73968)
Trial 22/30  평균AUC=0.73965  (현재 최고: 0.73968)
Trial 23/30  평균AUC=

## 4. 최종 검증: 기존 파라미터 vs 견고탐색 파라미터, 둘 다 10시드 배깅

In [4]:
def run_10seed_bagging(params, label):
    oof_per_seed = []
    for seed in FINAL_SEEDS:
        skf = StratifiedKFold(n_splits=FINAL_SPLITS, shuffle=True, random_state=seed)
        oof_seed = np.zeros(len(y))
        for tr_idx, val_idx in skf.split(X, y):
            m = LGBMClassifier(**params, random_state=seed, verbose=-1)
            m.fit(X.iloc[tr_idx], y[tr_idx])
            oof_seed[val_idx] = m.predict_proba(X.iloc[val_idx])[:, 1]
        oof_per_seed.append(oof_seed)

    oof_per_seed = np.array(oof_per_seed)
    individual_scores = np.array([roc_auc_score(y, oof_per_seed[i]) for i in range(len(FINAL_SEEDS))])
    oof_bagged = oof_per_seed.mean(axis=0)
    score_bagged = roc_auc_score(y, oof_bagged)

    print(f"[{label}] 시드별 AUC 평균: {individual_scores.mean():.5f}  표준편차: {individual_scores.std():.5f}")
    print(f"[{label}] 10시드 배깅 OOF AUC: {score_bagged:.5f}")
    return oof_bagged, score_bagged, individual_scores.std()


print("=== 기존 파라미터(단일시드 탐색) ===")
oof_old, score_old, std_old = run_10seed_bagging(old_best_params, "기존")

print()
print("=== 견고탐색 파라미터(3시드 탐색) ===")
oof_new, score_new, std_new = run_10seed_bagging(robust_best_params, "견고탐색")

=== 기존 파라미터(단일시드 탐색) ===
[기존] 시드별 AUC 평균: 0.74000  표준편차: 0.00011
[기존] 10시드 배깅 OOF AUC: 0.74036

=== 견고탐색 파라미터(3시드 탐색) ===
[견고탐색] 시드별 AUC 평균: 0.73997  표준편차: 0.00013
[견고탐색] 10시드 배깅 OOF AUC: 0.74037


## 5. 비교 및 결론

In [5]:
print("=" * 60)
print(f"기존 파라미터    : 배깅 AUC {score_old:.5f}  (시드간 표준편차 {std_old:.5f})")
print(f"견고탐색 파라미터: 배깅 AUC {score_new:.5f}  (시드간 표준편차 {std_new:.5f})")
print("=" * 60)

improvement = score_new - score_old
std_reduction = std_old - std_new

print(f"\nAUC 개선: {improvement:+.5f}")
print(f"표준편차 감소: {std_reduction:+.5f} (양수면 더 안정적이라는 뜻)")

if improvement > 0.001:
    print("\n=> 점수 자체가 노이즈를 넘는 수준으로 개선됐습니다! 견고탐색 파라미터로 교체를 검토하세요.")
elif std_reduction > 0.00005:
    print("\n=> 점수 개선은 작지만, 시드간 변동성이 줄었습니다. 더 안정적인 파라미터를 찾은 것으로 보입니다.")
else:
    print("\n=> 기존 파라미터와 큰 차이가 없습니다. 단일시드 탐색이 이미 충분히 견고했던 것으로 보입니다.")

기존 파라미터    : 배깅 AUC 0.74036  (시드간 표준편차 0.00011)
견고탐색 파라미터: 배깅 AUC 0.74037  (시드간 표준편차 0.00013)

AUC 개선: +0.00001
표준편차 감소: -0.00002 (양수면 더 안정적이라는 뜻)

=> 기존 파라미터와 큰 차이가 없습니다. 단일시드 탐색이 이미 충분히 견고했던 것으로 보입니다.


## 6. 결과 저장 (제출 후보, 더 좋은 쪽 채택)

In [6]:
final_params = robust_best_params if score_new >= score_old else old_best_params
final_oof = oof_new if score_new >= score_old else oof_old
final_label = "견고탐색" if score_new >= score_old else "기존"
print(f"채택: {final_label} 파라미터")

with open("../lgbm_robust_best_params.json", "w", encoding="utf-8") as f:
    json.dump(robust_best_params, f, ensure_ascii=False, indent=2)

np.save(f"{CACHE_DIR}/oof_lgbm_robust_search.npy", final_oof)
print(f"저장 완료: ../lgbm_robust_best_params.json, {CACHE_DIR}/oof_lgbm_robust_search.npy")
print("\n참고: test 예측 및 제출용 CSV가 필요하면 말씀해주세요 (현재는 OOF 검증까지만 수행).")

채택: 견고탐색 파라미터
저장 완료: ../lgbm_robust_best_params.json, ../blend_cache/oof_lgbm_robust_search.npy

참고: test 예측 및 제출용 CSV가 필요하면 말씀해주세요 (현재는 OOF 검증까지만 수행).
